In [1]:
#Creates a Cu vacancy and its corresponding 12 NN NEB images

import ase
import os
import sys
import random
import numpy as np
import copy
import shutil
sys.path.append("/home/gridsan/ebotica/3320")
from labutil.src.plugins.lammps import *
from ase import Atom
from ase.spacegroup import crystal
from ase.io import write
from ase.io import read
from ase.build import make_supercell
from ase.build import bulk
from ase.neighborlist import NeighborList
from ase.neighborlist import NewPrimitiveNeighborList
import matplotlib.pyplot as plt
from ase.calculators.vasp import Vasp



folder_name0='92.5Cu5Cr2.5Ta_Cu_NN_2'
folder_name='92.5Cu5Cr2.5Ta_Cu_NN_2'
folder_name_Cr='92.5Cu5Cr2.5Ta_Cr_NN'
folder_name_Ta='92.5Cu5Cr2.5Ta_Ta_NN'
os.environ['VASP_PP_PATH']='/home/gridsan/ebotica/IP_dev_shared/vasp'
os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name,exist_ok=True)


for j in range (10,15):
    print(folder_name, j)
    os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j),exist_ok=True)
    os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Cr+'/'+str(j),exist_ok=True)
    os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Ta+'/'+str(j),exist_ok=True)
    os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/vac_origin',exist_ok=True)
    #os.rmdir('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/relax0')
    #os.rmdir('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/relax1')
    supercell=read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/relax_pre/CONTCAR')
    supercell_1 = copy.deepcopy(supercell)
    
    # Step 2: Remove an atom to create a vacancy in the center of the supercell and just considering Cu atoms to be removed
    cu_pos=[]
    for i in range(0,len(supercell)):
        if supercell_1.symbols[i] == 'Cu':
            cu_pos.append(i)
        
    # Geometric center of the supercell
    cell_center = np.sum(supercell.get_cell(), axis=0) / 2
    distance = np.linalg.norm(supercell_1.positions - cell_center, axis=1)
    # Calculate distances from each atom to the geometric center
    dist_cu_pos=[]
    for i in cu_pos:
        dist_cu_pos.append(distance[i])

    vacancy_index=cu_pos[np.argmin(dist_cu_pos)]
    vacancy_position=supercell_1.positions[vacancy_index]

    print("supercell_1 vacancy pos:", supercell_1.positions[vacancy_index])
    supercell_1.pop(vacancy_index)  # Remove atom at index selected before
    #write supercell with a vacancy
    write(folder_name+'/'+str(j)+'/vac_origin/structure0_00.xyz', supercell_1)
    print("supercell_1:", supercell_1)
    #supercell_2 = copy.deepcopy(supercell_1) 
    # Step 3: Create VASP files
    # Step 3.1: Set up the Vasp calculator
    calc = Vasp(xc='PBE',  # Exchange-correlation functional
                setups='recommended',  # Use the recommended POTCAR files
                directory='/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/vac_origin',  # Output directory for VASP files
                txt='vasp_run.txt')  # Log file for the VASP run
    # Step 3.2: Attach the calculator to the atoms object
    supercell_1.calc = calc
    # Step 3.3: Write VASP input files (including POTCAR)
    calc.write_input(supercell_1)
    shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/INCAR_pNEB', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/vac_origin/INCAR')
    shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/KPOINTS', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/vac_origin/KPOINTS')   
    
    
    # Step 4: Remove an atom to create a vacancy in the center of the supercell, considering Cu atoms to be removed and the position on the previous vacancy to be neighbors
    cutoffs = [1.3] * len(supercell)
    nl = NewPrimitiveNeighborList(cutoffs, skin=0.3, self_interaction=False,bothways=True)
    # Update the neighbor list with the atoms to extract the info 
    nl.update(supercell.pbc, supercell.get_cell(complete=True),supercell.positions)

    # Get neighbors info of the target atom
    indices, offsets = nl.get_neighbors(vacancy_index)
    neighbor_info = []

    # Print the neighboring atoms' positions
    for i, offset in zip(indices, offsets):
        neighbor_position = supercell.positions[i] + offset @ supercell.get_cell()
        neighbor_symbols = supercell.symbols[i]
        distance = np.linalg.norm(neighbor_position - vacancy_position)
        print(f"Neighbor index: {i}, Position: {supercell.positions[i]}, Distance: {distance:.2f} Å, Atom type: {neighbor_symbols} ")
        neighbor_info.append((i, neighbor_position, distance, neighbor_symbols))
    
    cu_pos_2=[]
    cr_pos_2=[]
    ta_pos_2=[]
    for i in range(0,len(neighbor_info)):
        if neighbor_info[i][3] == 'Cu':
            cu_pos_2.append(neighbor_info[i][0])
    for ii in range(0,len(neighbor_info)):
        if neighbor_info[ii][3] == 'Cr':
            cr_pos_2.append(neighbor_info[ii][0])
    for iii in range(0,len(neighbor_info)):
        if neighbor_info[iii][3] == 'Ta':
            ta_pos_2.append(neighbor_info[iii][0])
    e=0
    for b in cu_pos_2:
        supercell_2 = copy.deepcopy(supercell_1)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e),exist_ok=True)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax0',exist_ok=True)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1',exist_ok=True)
        # Find which index correspond the selected atom from the pristine structure in the sructure with the vacancy
        for a in range(0,len(supercell_2)):
            if round(supercell_2.positions[a][0],4)==round(supercell.positions[b][0],4):
                if round(supercell_2.positions[a][1],4)==round(supercell.positions[b][1],4):
                    if round(supercell_2.positions[a][2],4)==round(supercell.positions[b][2],4):
                        new_vac_index=a
                        print(f'New vacancy index in supercell_2: {new_vac_index}')

        print("supercell_2 before:",supercell_2)
        print(new_vac_index,supercell_2[new_vac_index])
        print("supercell_2 vacancy pos:", supercell_2.positions[new_vac_index])
        supercell_2.positions[new_vac_index]= vacancy_position
        print("supercell_2 after:",supercell_2)

        write('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1/structure0_01.xyz', supercell_2)
        
        #Repeat for supercell_2
        calc = Vasp(xc='PBE',  # Exchange-correlation functional
                    setups='recommended',  # Use the recommended POTCAR files
                    directory='/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1',  # Output directory for VASP files
                    txt='vasp_run.txt')  # Log file for the VASP run
        supercell_2.calc = calc
        calc.write_input(supercell_2)
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/INCAR_pNEB', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1/INCAR')
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/KPOINTS', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1/KPOINTS')
        
        e=e+1
    
    ee=0
    for bb in cr_pos_2:
        supercell_2 = copy.deepcopy(supercell_1)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Cr+'/'+str(j)+'/'+str(ee),exist_ok=True)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Cr+'/'+str(j)+'/'+str(ee)+'/relax0',exist_ok=True)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Cr+'/'+str(j)+'/'+str(ee)+'/relax1',exist_ok=True)
        # Find which index correspond the selected atom from the pristine structure in the sructure with the vacancy
        for aa in range(0,len(supercell_2)):
            if round(supercell_2.positions[aa][0],4)==round(supercell.positions[bb][0],4):
                if round(supercell_2.positions[aa][1],4)==round(supercell.positions[bb][1],4):
                    if round(supercell_2.positions[aa][2],4)==round(supercell.positions[bb][2],4):
                        new_vac_index=aa
                        print(f'New vacancy index in supercell_2: {new_vac_index}')

        print("supercell_2 before:",supercell_2)
        print(new_vac_index,supercell_2[new_vac_index])
        print("supercell_2 vacancy pos:", supercell_2.positions[new_vac_index])
        supercell_2.positions[new_vac_index]= vacancy_position
        print("supercell_2 after:",supercell_2)

        write('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Cr+'/'+str(j)+'/'+str(ee)+'/relax1/structure0_01.xyz', supercell_2)
        
        #Repeat for supercell_2
        calc = Vasp(xc='PBE',  # Exchange-correlation functional
                setups='recommended',  # Use the recommended POTCAR files
                directory='/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Cr+'/'+str(j)+'/'+str(ee)+'/relax1',  # Output directory for VASP files
                txt='vasp_run.txt')  # Log file for the VASP run
        supercell_2.calc = calc
        calc.write_input(supercell_2)
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/INCAR_pNEB', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Cr+'/'+str(j)+'/'+str(ee)+'/relax1/INCAR')
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/KPOINTS', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Cr+'/'+str(j)+'/'+str(ee)+'/relax1/KPOINTS')
        ee=ee+1
    
    eee=0
    for bbb in ta_pos_2:
        supercell_2 = copy.deepcopy(supercell_1)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Ta+'/'+str(j)+'/'+str(eee),exist_ok=True)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Ta+'/'+str(j)+'/'+str(eee)+'/relax0',exist_ok=True)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Ta+'/'+str(j)+'/'+str(eee)+'/relax1',exist_ok=True)
        # Find which index correspond the selected atom from the pristine structure in the sructure with the vacancy
        for aaa in range(0,len(supercell_2)):
            if round(supercell_2.positions[aaa][0],4)==round(supercell.positions[bbb][0],4):
                if round(supercell_2.positions[aaa][1],4)==round(supercell.positions[bbb][1],4):
                    if round(supercell_2.positions[aaa][2],4)==round(supercell.positions[bbb][2],4):
                        new_vac_index=aaa
                        print(f'New vacancy index in supercell_2: {new_vac_index}')

        print("supercell_2 before:",supercell_2)
        print(new_vac_index,supercell_2[new_vac_index])
        print("supercell_2 vacancy pos:", supercell_2.positions[new_vac_index])
        supercell_2.positions[new_vac_index]= vacancy_position
        print("supercell_2 after:",supercell_2)

        write('/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Ta+'/'+str(j)+'/'+str(eee)+'/relax1/structure0_01.xyz', supercell_2)
        
        #Repeat for supercell_2
        calc = Vasp(xc='PBE',  # Exchange-correlation functional
                    setups='recommended',  # Use the recommended POTCAR files
                    directory='/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Ta+'/'+str(j)+'/'+str(eee)+'/relax1',  # Output directory for VASP files
                    txt='vasp_run.txt')  # Log file for the VASP run
        supercell_2.calc = calc
        calc.write_input(supercell_2)
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/INCAR_pNEB', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Ta+'/'+str(j)+'/'+str(eee)+'/relax1/INCAR')
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/KPOINTS', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name_Ta+'/'+str(j)+'/'+str(eee)+'/relax1/KPOINTS')
        eee=eee+1

92.5Cu5Cr2.5Ta_Cu_NN_2 10
supercell_1 vacancy pos: [5.53563481 3.70847365 9.14481412]
supercell_1: Atoms(symbols='Cr3Cu58Ta2', pbc=True, cell=[[0.020817318914747, 7.319331711500893, 7.338655712451575], [7.374014917981816, 0.0449819075790006, 7.3142951519224955], [7.390696229018537, 7.311481213434266, 0.0285521259942118]])
Neighbor index: 29, Position: [0.04086461 3.66845071 3.65384203], Distance: 2.62 Å, Atom type: Cu 
Neighbor index: 32, Position: [5.60348121 9.15590469 3.61069952], Distance: 2.53 Å, Atom type: Cu 
Neighbor index: 25, Position: [5.53536108 1.86018986 7.33843993], Distance: 2.58 Å, Atom type: Cu 
Neighbor index: 58, Position: [11.08106161 11.04200626 10.93921332], Distance: 2.55 Å, Atom type: Cu 
Neighbor index: 36, Position: [3.69445459 3.68379895 7.33306821], Distance: 2.58 Å, Atom type: Cu 
Neighbor index: 43, Position: [11.05378986  9.18803552  9.18843956], Distance: 2.62 Å, Atom type: Cu 
Neighbor index: 55, Position: [ 5.5426977   5.51798247 10.99615726], Distanc

In [6]:
##### CORRECTED - Busca un Cu en el centro y crea NEB images para todas las neighbor positions de ese Cu, luego descarta el inicial
import ase
import os
import sys
import random
import numpy as np
import copy
import shutil
sys.path.append("/home/gridsan/ebotica/3320")
from labutil.src.plugins.lammps import *
from ase import Atom
from ase.spacegroup import crystal
from ase.io import write
from ase.io import read
from ase.build import make_supercell
from ase.build import bulk
from ase.neighborlist import NeighborList
from ase.neighborlist import NewPrimitiveNeighborList
import matplotlib.pyplot as plt
from ase.calculators.vasp import Vasp



folder_name0='92.5Cu5Cr2.5Ta_Cu_NN_2'
folder_name='92.5Cu5Cr2.5Ta_Cu_NN_2'
os.environ['VASP_PP_PATH']='/home/gridsan/ebotica/IP_dev_shared/vasp'
os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name,exist_ok=True)

for j in range (10,15):
    print(folder_name, j)
    os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j),exist_ok=True)
        
    # Load the pristine and vacancy-containing structures
    vacancy_structure = read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/structure0_00.xyz')
    vacancy_structure_m = read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/structure0_01.xyz')
    pristine_structure = read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/relax_pre/CONTCAR')

    # Get the positions of atoms
    vacancy_positions = vacancy_structure.get_positions()
    vacancy_positions_m = vacancy_structure_m.get_positions()
    pristine_positions = pristine_structure.get_positions()
    supercell=copy.deepcopy(pristine_structure)
    print(supercell) 
    
    # Find the location of the original vacancy created on vacancy_positions and it's index on vacancy_positions_m
    missing_atom_index = 0
    for r in range(0, len(vacancy_positions_m)):
        if not np.array_equal(vacancy_positions_m[r], vacancy_positions[r]):
            print(vacancy_positions_m[r], vacancy_positions[r],r)
            missing_atom_index = r #position on vac_m that correspond with the location of the vacancy in vac
            break
    print(missing_atom_index)
    vacancy_index=missing_atom_index
    vacancy_position_ori = vacancy_positions_m[vacancy_index]
    vacancy_position_2 = vacancy_positions[vacancy_index]
    print(f'Original vacancy position: {vacancy_position_ori}')
    print(f'Second vacancy position: {vacancy_position_2}')
    

    # Find the location of the second vacancy created on vacancy_positions_m and it's index on vacancy_positions
 #   missing_atom_index_2 = 0
  #  for rr in range(0, len(vacancy_positions)):
   #     if not np.array_equal(vacancy_positions[rr], vacancy_positions_m[rr]):
    #        missing_atom_index_2 = rr #position on vac that correspond with the location of the vacancy in vac_m
    #        break
   # print(missing_atom_index_2)
   # vacancy_index_2=missing_atom_index_2
   # vacancy_position_2 = vacancy_positions[vacancy_index_2]
    
    
    # Find which index in the pristine structure correspond to the position of the vacancy that we already have
    for i in range(0,len(pristine_positions)):
        if round(pristine_structure.positions[i][0],4)==round(vacancy_position_ori[0],4):
            if round(pristine_structure.positions[i][1],4)==round(vacancy_position_ori[1],4):
                if round(pristine_structure.positions[i][2],4)==round(vacancy_position_ori[2],4):
                    vacancy_index_ori_pris=i
                    vacancy_position_ori_pris=pristine_structure[vacancy_index_ori_pris]
                    print(f'Original vacancy position pristine: {vacancy_position_ori_pris}')
                    
    # Find which index in the pristine structure correspond to the atom that it's going to be moving (the one to account for NN)
    for i in range(0,len(pristine_positions)):
        if round(pristine_structure.positions[i][0],4)==round(vacancy_position_2[0],4):
            if round(pristine_structure.positions[i][1],4)==round(vacancy_position_2[1],4):
                if round(pristine_structure.positions[i][2],4)==round(vacancy_position_2[2],4):
                    vacancy_index_2_pris=i
                    vacancy_position_2_pris=pristine_structure[vacancy_index_2_pris]
                    print(f'Second vacancy position pristine: {vacancy_position_2_pris}')
                    print(f'Existing vacancy position pristine: {vacancy_index_2_pris}')
    
    cutoffs = [1.3] * len(pristine_structure)
    nl = NewPrimitiveNeighborList(cutoffs, skin=0.3, self_interaction=False,bothways=True)
    # Update the neighbor list with the atoms to extract the info 
    nl.update(pristine_structure.pbc, pristine_structure.get_cell(complete=True),pristine_structure.positions)


    # Get neighbors info of the target atom
    indices, offsets = nl.get_neighbors(vacancy_index_ori_pris)



    neighbor_info = []

    # Print the neighboring atoms' positions
    for i, offset in zip(indices, offsets):
        neighbor_position = pristine_structure.positions[i] + offset @ pristine_structure.get_cell()
        neighbor_symbols = pristine_structure.symbols[i]
        distance = np.linalg.norm(neighbor_position - vacancy_position_ori)
        print(f"Neighbor index: {i}, Position: {pristine_structure.positions[i]}, Distance: {distance:.2f} Å, Atom type: {neighbor_symbols} ")
        neighbor_info.append((i, neighbor_position, distance, neighbor_symbols))

    cu_pos_2=[]   
    for i in range(0,len(neighbor_info)):
        if neighbor_info[i][3] == 'Cu' and not neighbor_info[i][0]==vacancy_index_2_pris:
            cu_pos_2.append(neighbor_info[i][0])
    e=0
    source_relax0 = '/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/relax0'
    
    for b in cu_pos_2:
        supercell_2 = copy.deepcopy(vacancy_structure)
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e),exist_ok=True)
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/INCAR_NEB1', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/INCAR')
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/KPOINTS', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/KPOINTS')
        os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1',exist_ok=True)
        # Find which index correspond the selected atom from the pristine structure in the sructure with the vacancy
        for a in range(0,len(supercell_2)):
            if round(supercell_2.positions[a][0],4)==round(pristine_structure.positions[b][0],4):
                if round(supercell_2.positions[a][1],4)==round(pristine_structure.positions[b][1],4):
                    if round(supercell_2.positions[a][2],4)==round(pristine_structure.positions[b][2],4):
                        new_vac_index=a
                        print(f'New vacancy index in supercell_2: {new_vac_index}')

        print("supercell_2 before:",supercell_2)
        print(new_vac_index,supercell_2[new_vac_index])
        print("supercell_2 vacancy pos:", supercell_2.positions[new_vac_index])
        supercell_2.positions[new_vac_index]= vacancy_position_ori
        print("supercell_2 after:",supercell_2)

        write('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/structure0_01.xyz', supercell_2)
        
        #Repeat for supercell_2
        calc = Vasp(xc='PBE',  # Exchange-correlation functional
                    setups='recommended',  # Use the recommended POTCAR files
                    directory='/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1',  # Output directory for VASP files
                    txt='vasp_run.txt')  # Log file for the VASP run
        supercell_2.calc = calc
        calc.write_input(supercell_2)    
        relax0_dir = '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax0'
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/structure0_00.xyz', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/structure0_00.xyz')
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/INCAR_pNEB', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1/INCAR')
        shutil.copy2('/home/gridsan/ebotica/IPdev/NEB/KPOINTS', '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1/KPOINTS')
        os.makedirs(relax0_dir,exist_ok=True)
        files_2_copy=['INCAR','POSCAR','POTCAR','KPOINTS','CONTCAR','OUTCAR','OSZICAR']
        for filename in files_2_copy:
            source_file = os.path.join(source_relax0, filename)
            destination_file = os.path.join(relax0_dir, filename)
            shutil.copy2(source_file, destination_file)
        e=e+1

92.5Cu5Cr2.5Ta_Cu_NN_2 0
Atoms(symbols='Cr3Cu59Ta2', pbc=True, cell=[[0.0050490580725959, 7.345471343982061, 7.376495066600339], [7.243202576899881, -0.0029438890198006, 7.3845944230955025], [7.238981241913802, 7.349282212901479, 0.0013699614856523]])
[9.01246693 5.52454896 7.38971174] [10.83882357  7.33829911  7.38352211] 28
28
Original vacancy position: [9.01246693 5.52454896 7.38971174]
Second vacancy position: [10.83882357  7.33829911  7.38352211]
Original vacancy position pristine: Atom('Cu', [9.012466932934682, 5.524548958698128, 7.389711735204237], index=28)
Second vacancy position pristine: Atom('Cu', [10.838823566045367, 7.338299106285796, 7.383522109675499], index=29)
Existing vacancy position pristine: 29
Neighbor index: 16, Position: [3.60138711 5.53775683 1.87688614], Distance: 2.62 Å, Atom type: Cu 
Neighbor index: 27, Position: [7.22083727 3.68342117 7.38398879], Distance: 2.57 Å, Atom type: Cu 
Neighbor index: 59, Position: [3.60775818 3.69351468 0.00485068], Distance: 

In [4]:
##### WRONGBusca un Cu en el centro y crea NEB images para todas las neighbor positions de ese Cu, luego descarta el inicial
import ase
import os
import sys
import random
import numpy as np
import copy
import shutil
sys.path.append("/home/gridsan/ebotica/3320")
from labutil.src.plugins.lammps import *
from ase import Atom
from ase.spacegroup import crystal
from ase.io import write
from ase.io import read
from ase.build import make_supercell
from ase.build import bulk
from ase.neighborlist import NeighborList
from ase.neighborlist import NewPrimitiveNeighborList
import matplotlib.pyplot as plt
from ase.calculators.vasp import Vasp



folder_name0='71Cu24Cr5Ta_Cu'
folder_name='71Cu24Cr5Ta_Cu_NN_test'
os.environ['VASP_PP_PATH']='/home/gridsan/ebotica/IP_dev_shared/vasp'
os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name,exist_ok=True)

for j in range (0,10):
    print(folder_name, j)
    os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j),exist_ok=True)
        
    # Load the pristine and vacancy-containing structures
    vacancy_structure = read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/structure0_00.xyz')
    vacancy_structure_m = read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/structure0_01.xyz')
    pristine_structure = read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/relax_pre/CONTCAR')

    # Get the positions of atoms
    vacancy_positions = vacancy_structure.get_positions()
    vacancy_positions_m = vacancy_structure_m.get_positions()
    pristine_positions = pristine_structure.get_positions()
    supercell=copy.deepcopy(pristine_structure)
    print(supercell) 
    
    # Find the missing atom
    missing_atom_index = 0
    for r in range(0, len(vacancy_positions_m)):
        if not np.array_equal(vacancy_positions_m[r], vacancy_positions[r]):
            missing_atom_index = r
            break
    print(missing_atom_index)
    vacancy_index=missing_atom_index
    atom_NN_position = vacancy_positions_m[vacancy_index]
    vacancy_position = vacancy_positions[vacancy_index]
    
    
    # Find which index in the pristine structure correspond to the atom that it's going to be moving (the one to account for NN)
    for i in range(0,len(pristine_positions)):
        if round(pristine_structure.positions[i][0],4)==round(atom_NN_position[0],4):
            if round(pristine_structure.positions[i][1],4)==round(atom_NN_position[1],4):
                if round(pristine_structure.positions[i][2],4)==round(atom_NN_position[2],4):
                    atom_NN_index_pris=i
                    print(f'Atom for NN: {atom_NN_index_pris}')
                    
    # Find which index in the pristine structure correspond to the position of the vacancy that we already have
    for i in range(0,len(pristine_positions)):
        if round(pristine_structure.positions[i][0],4)==round(vacancy_position[0],4):
            if round(pristine_structure.positions[i][1],4)==round(vacancy_position[1],4):
                if round(pristine_structure.positions[i][2],4)==round(vacancy_position[2],4):
                    vac_index_pris=i
                    print(f'Existing_vacancy: {vac_index_pris}')
                    
    # Step 4: Remove an atom to create a vacancy in the center of the supercell, considering Cu atoms to be removed and the position on the previous vacancy to be neighbors
    cutoffs = [1.3] * len(supercell)
    #nl = NewPrimitiveNeighborList(cutoffs, skin=0.3, self_interaction=False,bothways=True)
    nl = NeighborList(cutoffs, skin=0.3, self_interaction=False, bothways=True)
    # Update the neighbor list with the atoms to extract the info 
    #nl.update(supercell.pbc, supercell.get_cell(complete=True),supercell.positions)
    nl.update(supercell)

    # Get neighbors info of the target atom
    indices, offsets = nl.get_neighbors(atom_NN_index_pris)
    neighbor_info = []
    target_position = supercell.positions[atom_NN_index_pris]
    target_position
    print(indices)
    
    e=0
    source_relax0 = '/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/relax0'
    
    # Print the neighboring atoms' positions
    for i, offset in zip(indices, offsets):
        supercell=copy.deepcopy(pristine_structure)
        if supercell.symbols[i]=='Cu'and not i==vac_index_pris:
            os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1',exist_ok=True)
            neighbor_position = supercell.positions[i] + offset @ supercell.get_cell()
            neighbor_symbols = supercell.symbols[i]
            distance = np.linalg.norm(neighbor_position - target_position)
            print(f"Neighbor index: {i}, Position: {supercell.positions[i]}, Distance: {distance:.2f} Å, Atom type: {neighbor_symbols} ")
            neighbor_info.append((i, neighbor_position, distance, neighbor_symbols))
            supercell.pop(i) 
            write(folder_name+'/'+str(j)+'/'+str(e)+'/structure0_01.xyz', supercell)
            calc = Vasp(xc='PBE',  # Exchange-correlation functional
                    setups='recommended',  # Use the recommended POTCAR files
                    directory='/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1',  # Output directory for VASP files
                    txt='vasp_run.txt')  # Log file for the VASP run
            supercell.calc = calc
            calc.write_input(supercell)
            relax0_dir = '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax0'
            os.makedirs(relax0_dir,exist_ok=True)
            files_2_copy=['INCAR','POSCAR','POTCAR','KPOINTS','CONTCAR','OUTCAR','OSZICAR']
            for filename in files_2_copy:
                source_file = os.path.join(source_relax0, filename)
                destination_file = os.path.join(relax0_dir, filename)
                shutil.copy2(source_file, destination_file)
            e=e+1

KeyboardInterrupt: 

In [1]:
##### Busca un Cu en el centro y crea NEB images para todas las neighbor positions de ese Cu que sean Ta
import ase
import os
import sys
import random
import numpy as np
import copy
import shutil
sys.path.append("/home/gridsan/ebotica/3320")
from labutil.src.plugins.lammps import *
from ase import Atom
from ase.spacegroup import crystal
from ase.io import write
from ase.io import read
from ase.build import make_supercell
from ase.build import bulk
from ase.neighborlist import NeighborList
from ase.neighborlist import NewPrimitiveNeighborList
import matplotlib.pyplot as plt
from ase.calculators.vasp import Vasp



folder_name0='87Cu9Cr4Ta_Cu'
folder_name='87Cu9Cr4Ta_Ta_NN'
os.environ['VASP_PP_PATH']='/home/gridsan/ebotica/IP_dev_shared/vasp'
os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name,exist_ok=True)

for j in range (0,10):
    print(folder_name, j)
    os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j),exist_ok=True)
        
    # Load the pristine and vacancy-containing structures
    vacancy_structure = read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/structure0_00.xyz')
    vacancy_structure_m = read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/structure0_01.xyz')
    pristine_structure = read('/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/relax_pre/CONTCAR')

    # Get the positions of atoms
    vacancy_positions = vacancy_structure.get_positions()
    vacancy_positions_m = vacancy_structure_m.get_positions()
    pristine_positions = pristine_structure.get_positions()
    supercell=copy.deepcopy(pristine_structure)
    print(supercell) 
    
    # Find the missing atom
    missing_atom_index = 0
    for r in range(0, len(vacancy_positions_m)):
        if not np.array_equal(vacancy_positions_m[r], vacancy_positions[r]):
            missing_atom_index = r
            break
    print(missing_atom_index)
    vacancy_index=missing_atom_index
    atom_NN_position = vacancy_positions_m[vacancy_index]
    vacancy_position = vacancy_positions[vacancy_index]
    
    
    # Find which index in the pristine structure correspond to the atom that it's going to be moving (the one to account for NN)
    for i in range(0,len(pristine_positions)):
        if round(pristine_structure.positions[i][0],4)==round(atom_NN_position[0],4):
            if round(pristine_structure.positions[i][1],4)==round(atom_NN_position[1],4):
                if round(pristine_structure.positions[i][2],4)==round(atom_NN_position[2],4):
                    atom_NN_index_pris=i
                    print(f'Atom for NN: {atom_NN_index_pris}')
                    
    # Find which index in the pristine structure correspond to the position of the vacancy that we already have
    for i in range(0,len(pristine_positions)):
        if round(pristine_structure.positions[i][0],4)==round(vacancy_position[0],4):
            if round(pristine_structure.positions[i][1],4)==round(vacancy_position[1],4):
                if round(pristine_structure.positions[i][2],4)==round(vacancy_position[2],4):
                    vac_index_pris=i
                    print(f'Existing_vacancy: {vac_index_pris}')
                    
    # Step 4: Remove an atom to create a vacancy in the center of the supercell, considering Cu atoms to be removed and the position on the previous vacancy to be neighbors
    cutoffs = [1.3] * len(supercell)
    #nl = NewPrimitiveNeighborList(cutoffs, skin=0.3, self_interaction=False,bothways=True)
    nl = NeighborList(cutoffs, skin=0.3, self_interaction=False, bothways=True)
    # Update the neighbor list with the atoms to extract the info 
    #nl.update(supercell.pbc, supercell.get_cell(complete=True),supercell.positions)
    nl.update(supercell)

    # Get neighbors info of the target atom
    indices, offsets = nl.get_neighbors(atom_NN_index_pris)
    neighbor_info = []
    target_position = supercell.positions[atom_NN_index_pris]
    target_position
    print(indices)
    
    e=0
    source_relax0 = '/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/'+str(j)+'/relax0'
    
    # Print the neighboring atoms' positions
    for i, offset in zip(indices, offsets):
        supercell=copy.deepcopy(pristine_structure)
        if supercell.symbols[i]=='Ta'and not i==vac_index_pris:
            os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1',exist_ok=True)
            neighbor_position = supercell.positions[i] + offset @ supercell.get_cell()
            neighbor_symbols = supercell.symbols[i]
            distance = np.linalg.norm(neighbor_position - target_position)
            print(f"Neighbor index: {i}, Position: {supercell.positions[i]}, Distance: {distance:.2f} Å, Atom type: {neighbor_symbols} ")
            neighbor_info.append((i, neighbor_position, distance, neighbor_symbols))
            supercell[vac_index_pris].symbol='Ta'
            supercell.pop(i)
            write(folder_name+'/'+str(j)+'/'+str(e)+'/structure0_01.xyz', supercell)
            calc = Vasp(xc='PBE',  # Exchange-correlation functional
                    setups='recommended',  # Use the recommended POTCAR files
                    directory='/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax1',  # Output directory for VASP files
                    txt='vasp_run.txt')  # Log file for the VASP run
            supercell.calc = calc
            calc.write_input(supercell)
            relax0_dir = '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax0'
            os.makedirs(relax0_dir,exist_ok=True)
            files_2_copy=['INCAR','POSCAR','POTCAR','KPOINTS','CONTCAR','OUTCAR','OSZICAR']
            for filename in files_2_copy:
                source_file = os.path.join(source_relax0, filename)
                destination_file = os.path.join(relax0_dir, filename)
                shutil.copy2(source_file, destination_file)
            e=e+1

87Cu9Cr4Ta_Ta_NN 0
Atoms(symbols='Cr6Cu56Ta2', pbc=True, cell=[[-0.0367074798608183, 7.403336821480275, 7.278447296152704], [7.274590477325886, 0.0085626183241272, 7.233708439910969], [7.308696588126428, 7.392205219262286, -0.0256733810973182]])
23
Atom for NN: 35
Existing_vacancy: 23
[46 59 49 50 36 22 23 25 33 34 37 38]
87Cu9Cr4Ta_Ta_NN 1
Atoms(symbols='Cr6Cu56Ta2', pbc=True, cell=[[0.0438881548782731, 7.267143470685845, 7.367294423279993], [7.3883120075012165, 0.0204522297672076, 7.390522335502676], [7.359441303898074, 7.261524224009136, 0.049568879598997]])
38
Atom for NN: 35
Existing_vacancy: 39
[46 21 22 24 32 33 34 36 38 39 49 50]
87Cu9Cr4Ta_Ta_NN 2
Atoms(symbols='Cr6Cu56Ta2', pbc=True, cell=[[-0.0188949063148432, 7.245428012924105, 7.3364716253734805], [7.407184393700729, -0.0049475516309176, 7.322926568022324], [7.39314756744078, 7.218374411950506, 0.0087037455089684]])
48
Atom for NN: 37
Existing_vacancy: 49
[61 40 50 49 23 24 27 33 34 36 38 41]
87Cu9Cr4Ta_Ta_NN 3
Atoms(symbo

In [4]:
##### Random
import ase
import os
import sys
import random
import numpy as np
import copy
import shutil
sys.path.append("/home/gridsan/ebotica/3320")
from labutil.src.plugins.lammps import *
from ase import Atom
from ase.spacegroup import crystal
from ase.io import write
from ase.io import read
from ase.build import make_supercell
from ase.build import bulk
from ase.neighborlist import NeighborList
from ase.neighborlist import NewPrimitiveNeighborList
import matplotlib.pyplot as plt
from ase.calculators.vasp import Vasp



folder_name0='71Cu24Cr5Ta_Cu_0'
folder_name='71Cu24Cr5Ta_Cu_NN'
os.environ['VASP_PP_PATH']='/home/gridsan/ebotica/IP_dev_shared/vasp'
os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name,exist_ok=True)


for j in range (0,1):
    print(folder_name, j)
    os.makedirs('/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j),exist_ok=True)
    e=0
    source_relax0 = '/home/gridsan/ebotica/IPdev/NEB/'+folder_name0+'/0/relax0'
    for e in range(0,4):
        relax0_dir = '/home/gridsan/ebotica/IPdev/NEB/'+folder_name+'/'+str(j)+'/'+str(e)+'/relax0'
        files_2_copy=['INCAR','POSCAR','POTCAR','KPOINTS','CONTCAR','OUTCAR','OSZICAR']
        for filename in files_2_copy:
            source_file = os.path.join(source_relax0, filename)
            destination_file = os.path.join(relax0_dir, filename)
            shutil.copy2(source_file, destination_file)
        
  

71Cu24Cr5Ta_Cu_NN 0
